In [1]:
import pandas as pd 
import numpy as np
from dk_model import DeepKrigingTrainer

In [2]:
deposit_data = pd.read_csv("C:/Users/lin236/OneDrive - CSIRO/Minerals4D/Projects/SandS_DS/Parker Challenge 2025/filtered_deposit_data.csv", low_memory=False)
deposit_data

,X.1,HOLEID,SAMPFROM_1m,SAMPTO_1m,SAMPLETYPE,Au_ppm,LITH_INTERP,LITH_LOGGED,Ag_ppm_BESTEL,Al_pct_BESTEL,...,Y_ppm_BESTEL,Yb_ppm_BESTEL,Zn_ppm_BESTEL,Zr_ppm_BESTEL,X,Y,Z,Log_Au_ppm,Log_As_ppm_BESTEL,Log_Hg_ppm_BESTEL
0,56662,BD-022,0,1,RC,0.000354,NaN,DSOup_Ov,0.77,1.11,...,7.5,NaN,50.0,32.6,0.574274,0.636578,0.585957,-3.352407,4.123903,-0.916291
1,56664,BD-022,2,3,RC,0.000354,Ovi,DSOup_Ov,0.77,1.11,...,7.5,NaN,50.0,32.6,0.574276,0.636571,0.585614,-3.352407,4.123903,-0.916291
2,56666,BD-022,4,5,RC,0.000354,Ovi,DSOup_Ov,0.77,1.11,...,7.5,NaN,50.0,32.6,0.574279,0.636565,0.585271,-3.352407,4.123903,-0.916291
3,56668,BD-022,6,7,RC,0.000407,Ovi,DSOup_Ov,0.77,1.11,...,7.5,NaN,50.0,32.6,0.574281,0.636559,0.584928,-3.218876,4.123903,-0.916291
4,56670,BD-022,8,9,RC,0.000407,Ovi,DSOup_Ov,0.77,1.11,...,7.5,NaN,50.0,32.6,0.574283,0.636552,0.584585,-3.218876,4.123903,-0.916291
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92280,1059799,S4-280-2,92,93,CORE,0.000911,Drc,Bx Generic,NaN,NaN,...,NaN,NaN,NaN,NaN,0.358010,0.651042,0.398598,-2.441847,5.003946,-0.837018
92281,1059801,S4-280-2,94,95,CORE,0.003226,Drc,Bx Generic,NaN,NaN,...,NaN,NaN,NaN,NaN,0.357674,0.651101,0.398562,-1.194022,5.003946,-0.837018
92282,1059803,S4-280-2,96,97,CORE,0.003226,Drc,Bx Generic,NaN,NaN,...,NaN,NaN,NaN,NaN,0.357338,0.651160,0.398526,-1.194022,5.003946,-0.837018
92283,1059805,S4-280-2,98,99,CORE,0.009111,Drc,Bx Generic,NaN,NaN,...,NaN,NaN,NaN,NaN,0.357002,0.651220,0.398491,-0.160169,5.003946,-0.837018


In [3]:
deposit_data.values[:, 5]

array([0.0003537053313039, 0.0003537053313039, 0.0003537053313039, ...,
       0.0032262213552273, 0.009110591866921, 0.009110591866921],
      dtype=object)

In [4]:
N = len(deposit_data)

lon = deposit_data.values[:, 82]
lat = deposit_data.values[:, 83]
az = deposit_data.values[:, 84]
'''
num_basis_3_lvl = [10**3, 19**3, 37**3]
num_basis_2_lvl = [10**3, 19**3]
num_basis_1_lvl = [10**3]
'''

num_basis_3_lvl = [5**3, 10**3, 18**3]
num_basis_2_lvl = [5**3, 10**3]
num_basis_1_lvl = [5**3]

num_basis_list = [num_basis_3_lvl, num_basis_2_lvl, num_basis_1_lvl]

phi_arrays = []  

# For each grid
for grid in num_basis_list:
    knots_1dx = [np.linspace(0, 1, int(i**(1/3)) + 1) for i in grid]
    knots_1dy = [np.linspace(0, 1, int(i**(1/3)) + 1) for i in grid]
    knots_1dz = [np.linspace(0, 1, int(i**(1/3)) + 1) for i in grid]
    basis_size = 0
    phis = np.zeros((N, sum(grid)))
    
    # For each level of resolution
    for res in range(len(grid)):
        theta = 1 / (grid[res]**(1/3)) * 2.5
        knots_x, knots_y, knots_z = np.meshgrid(knots_1dx[res], knots_1dy[res], knots_1dz[res])
        knots = np.column_stack((knots_x.flatten(), knots_y.flatten(), knots_z.flatten()))
        
        # For each node in the grid
        for i in range(grid[res]):
            d = np.linalg.norm(np.vstack((lon, lat, az)).astype(float).T - knots[i, :], axis=1) / theta
            
            # For each distance of our data to the node i, calculate Wendland kernel
            for j in range(len(d)):
                if 0 <= d[j] <= 1:
                    phis[j, i + basis_size] = (1 - d[j])**6 * (35 * d[j]**2 + 18 * d[j] + 3) / 3
                else:
                    phis[j, i + basis_size] = 0
        
        basis_size += grid[res]
    
    phi_arrays.append(phis)  # Store the phi array for this grid level

# Unpack phi arrays into individual variables
phi_1_lvl, phi_2_lvl, phi_3_lvl = phi_arrays


In [5]:
phis = [phi_1_lvl, phi_2_lvl, phi_3_lvl]
phi_reduces = {}
dfs = []

#phi_columns = deposit_data.columns[10:].tolist()
phi_columns = deposit_data.columns[5:].tolist()

# Display the list of column names
#print(phi_columns[:10])
print(phi_columns[:5])

#total_columns = ['CP_Total', 'PO_Total', 'PY_Total']
total_columns = ['As_ppm_BESTEL', 'Hg_ppm_BESTEL']

# All covariates
#covariates = total_columns[:3] + ['RQD_Pct', 'Cr_ppm'] 
covariates = total_columns[-2:]


deposit_data_f = deposit_data.dropna(subset=['Au_ppm'] + covariates) # + phi_columns)

for idx, phi in enumerate(phis, start=1):
    idx_zero = np.array([], dtype=int)
    for i in range(phi.shape[1]):
        if np.sum(phi[:, i] != 0) == 0:
            idx_zero = np.append(idx_zero, int(i))

    phi_reduce = np.delete(phi, idx_zero, 1)
    phi_reduces[f"phi_{idx}_lvl_reduce"] = phi_reduce
    
    len_phi_regular = phi.shape[1]
    df_phi_regular = pd.DataFrame(phi, columns=[f'phi_{i}' for i in range(len_phi_regular)])
    dfs.append(df_phi_regular)
    
    len_phi_reduce = phi_reduce.shape[1]
    df_phi_reduce = pd.DataFrame(phi_reduce, columns=[f'phi_{i}' for i in range(len_phi_reduce)])
    dfs.append(df_phi_reduce)
    


['Au_ppm', 'LITH_INTERP', 'LITH_LOGGED', 'Ag_ppm_BESTEL', 'Al_pct_BESTEL']


In [6]:
deposit_data_list = []
for df in dfs:
    df_reset = df.reset_index(drop=True)
    deposit_data_reset = deposit_data_f.reset_index(drop=True)

    # Concatenate along columns
    deposit_data_basis = pd.concat([deposit_data_reset, df], axis=1)
    #phi_columns = deposit_data_basis.columns[10:].tolist()
    #total_columns = ['CP_Total','PO_Total', 'PY_Total']
    total_columns = ['As_ppm_BESTEL', 'Hg_ppm_BESTEL']
    #covariates = total_columns[:3] + ['RQD_Pct', 'Cr_ppm'] 
    covariates = total_columns[-2:]
    deposit_data_basis = deposit_data_basis.dropna(subset=['Au_ppm'] + covariates) # + phi_columns)

    deposit_data_list.append(deposit_data_basis)

## Comparison varying the levels of the basis function generating grid

In [7]:
dfs_names = ['3 levels', '3 levels no 0s', '2 levels', '2 levels no 0s', '1 level', '1 level no 0s']
for df, df_name in zip(deposit_data_list[1:], dfs_names[1:]):
    print(f"\nMetrics for df with {len(df.columns)} columns (grid with {df_name})")
    if df.empty:
        print(f"\nDataFrame for {df_name} is empty. Skipping...")
        continue
    trainer = DeepKrigingTrainer(df, regular_nn=False, plot_errors=False)
    trainer.train_neural_network()



Metrics for df with 2896 columns (grid with 3 levels no 0s)


ValueError: Input contains NaN.

In [8]:
#Choose the last one
deposit_data_list[-1].to_csv('Data/final_dataset_1_no_0.csv', index=False)